In [1]:
from pathlib import Path
import pandas as pd
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import GroupKFold

def set_seed(seed: int = 22):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(27)

DATA_DIR = Path('/kaggle/input/competitions/champs-scalar-coupling')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
structures = pd.read_csv(DATA_DIR / 'structures.csv')
dipole = pd.read_csv(DATA_DIR / 'dipole_moments.csv')
mulliken_charges = pd.read_csv(DATA_DIR / 'mulliken_charges.csv')
potential_energy = pd.read_csv(DATA_DIR / 'potential_energy.csv')
magnetic_shielding_tensors = pd.read_csv(DATA_DIR / 'magnetic_shielding_tensors.csv')

print('train:', train.shape)
print('test:', test.shape)
print('structures:', structures.shape)
print('train columns:', train.columns.tolist())

train: (4659076, 6)
test: (2505190, 5)
structures: (2358875, 6)
train columns: ['id', 'molecule_name', 'atom_index_0', 'atom_index_1', 'type', 'scalar_coupling_constant']


In [2]:
train.head()

,id,molecule_name,atom_index_0,atom_index_1,type,scalar_coupling_constant
0,0,dsgdb9nsd_000001,1,0,1JHC,84.8076
1,1,dsgdb9nsd_000001,1,2,2JHH,-11.2570
2,2,dsgdb9nsd_000001,1,3,2JHH,-11.2548
3,3,dsgdb9nsd_000001,1,4,2JHH,-11.2543
4,4,dsgdb9nsd_000001,2,0,1JHC,84.8074


In [3]:
structures.head()

,molecule_name,atom_index,atom,x,y,z
0,dsgdb9nsd_000001,0,C,-0.012698,1.085804,0.008001
1,dsgdb9nsd_000001,1,H,0.002150,-0.006031,0.001976
2,dsgdb9nsd_000001,2,H,1.011731,1.463751,0.000277
3,dsgdb9nsd_000001,3,H,-0.540815,1.447527,-0.876644
4,dsgdb9nsd_000001,4,H,-0.523814,1.437933,0.906397


In [4]:
splits=GroupKFold(n_splits=5)
train['fold']=-999
for fold, (train_idx, val_idx) in enumerate(splits.split(train, groups=train['molecule_name'])):
    train.loc[val_idx, 'fold'] = fold

In [5]:
train_fold = train[train['fold'] != 0].copy()
val_fold = train[train['fold'] == 0].copy()

## Feature engineering

In [6]:
element_counts = pd.crosstab(
    structures['molecule_name'], structures['atom']
).reindex(columns=['H', 'C', 'N', 'O', 'F'], fill_value=0).add_prefix('count_')
element_counts['n_atoms'] = element_counts.sum(axis=1)
element_counts.head()

atom,count_H,count_C,count_N,count_O,count_F,n_atoms
molecule_name,,,,,,
dsgdb9nsd_000001,4,1,0,0,0,5
dsgdb9nsd_000002,3,0,1,0,0,4
dsgdb9nsd_000003,2,0,0,1,0,3
dsgdb9nsd_000004,2,2,0,0,0,4
dsgdb9nsd_000005,1,1,1,0,0,3


In [7]:
atom_features = structures.merge(
    mulliken_charges,
    on=['molecule_name', 'atom_index'],
    how='left', validate='one_to_one'
).merge(
    magnetic_shielding_tensors.assign(
        shielding=(magnetic_shielding_tensors['XX']
                   + magnetic_shielding_tensors['YY']
                   + magnetic_shielding_tensors['ZZ']) / 3
    )[['molecule_name', 'atom_index', 'shielding']],
    on=['molecule_name', 'atom_index'],
    how='left', validate='one_to_one'
)
atom_features

,molecule_name,atom_index,atom,x,y,z,mulliken_charge,shielding
0,dsgdb9nsd_000001,0,C,-0.012698,1.085804,0.008001,-0.535689,195.316233
1,dsgdb9nsd_000001,1,H,0.002150,-0.006031,0.001976,0.133921,31.460567
2,dsgdb9nsd_000001,2,H,1.011731,1.463751,0.000277,0.133922,31.460567
3,dsgdb9nsd_000001,3,H,-0.540815,1.447527,-0.876644,0.133923,31.460633
4,dsgdb9nsd_000001,4,H,-0.523814,1.437933,0.906397,0.133923,31.460600
...,...,...,...,...,...,...,...,...
2358870,dsgdb9nsd_133885,11,H,-1.454004,-0.967309,1.459246,0.078144,28.796500
2358871,dsgdb9nsd_133885,12,H,0.277779,-2.697872,0.195770,0.099332,30.025400
2358872,dsgdb9nsd_133885,13,H,2.515854,-1.151784,0.527369,0.114018,27.196333
2358873,dsgdb9nsd_133885,14,H,0.013699,1.199431,-1.680192,0.078144,28.796533


In [8]:
molecule_features = potential_energy.merge(
    dipole.assign(
        dipole_magnitude=np.sqrt(dipole['X'] ** 2 + dipole['Y'] ** 2 + dipole['Z'] ** 2)
    )[['molecule_name', 'dipole_magnitude']],
    on='molecule_name', how='left', validate='one_to_one'
)
molecule_features

,molecule_name,potential_energy,dipole_magnitude
0,dsgdb9nsd_000001,-40.523680,0.000000
1,dsgdb9nsd_000002,-56.560246,1.625600
2,dsgdb9nsd_000003,-76.426077,1.851100
3,dsgdb9nsd_000004,-77.335268,0.000000
4,dsgdb9nsd_000005,-93.428488,2.893700
...,...,...,...
130784,dsgdb9nsd_133881,-400.761274,1.663671
130785,dsgdb9nsd_133882,-400.757209,1.297665
130786,dsgdb9nsd_133883,-380.894376,1.248000
130787,dsgdb9nsd_133884,-364.872596,1.957584


In [9]:
def add_features(data):
    for i in [0, 1]:
        data = data.merge(
            atom_features.rename(columns={
                'atom_index': f'atom_index_{i}',
                'atom': f'atom_{i}',
                'mulliken_charge': f'charge_{i}',
                'shielding': f'shielding_{i}',
                'x': f'x_{i}', 'y': f'y_{i}', 'z': f'z_{i}'
            }),
            on=['molecule_name', f'atom_index_{i}'],
            how='left',
            sort=False,
            validate='many_to_one'
        )

    data['distance'] = np.sqrt(
        (data['x_0'] - data['x_1']) ** 2
        + (data['y_0'] - data['y_1']) ** 2
        + (data['z_0'] - data['z_1']) ** 2
    )
    data['inverse_distance'] = 1 / data['distance'].clip(lower=1e-6)
    data['inverse_distance_squared'] = data['inverse_distance'] ** 2

    data['charge_difference'] = data['charge_0'] - data['charge_1']
    data['shielding_difference'] = data['shielding_0'] - data['shielding_1']
    data = data.merge(
        molecule_features, on='molecule_name', how='left',
        sort=False, validate='many_to_one'
    ).join(element_counts, on='molecule_name')

    missing = data[[
        'charge_0', 'charge_1', 'shielding_0', 'shielding_1',
        'potential_energy', 'dipole_magnitude'
    ]].isna().sum()
    if missing.any():
        raise ValueError(f'Missing property values after merging:\n{missing[missing > 0]}')

    return data


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), [
        'distance', 'inverse_distance', 'inverse_distance_squared',
        'count_H', 'count_C', 'count_N', 'count_O', 'count_F', 'n_atoms',
        'potential_energy', 'dipole_magnitude',
        'charge_0', 'charge_1', 'charge_difference',
        'shielding_0', 'shielding_1', 'shielding_difference'
    ]),
    ('categorical', OneHotEncoder(
        handle_unknown='ignore', sparse_output=False
    ), ['type', 'atom_0', 'atom_1'])
])

X_train = preprocessor.fit_transform(add_features(train_fold)).astype(np.float32)
X_valid = preprocessor.transform(add_features(val_fold)).astype(np.float32)
X_test = preprocessor.transform(add_features(test)).astype(np.float32)

y_train = train_fold['scalar_coupling_constant'].to_numpy(dtype=np.float32)
y_valid = val_fold['scalar_coupling_constant'].to_numpy(dtype=np.float32)

print('Train:', X_train.shape)
print('Validation:', X_valid.shape)
print('Test:', X_test.shape)


Train: (3727261, 29)
Validation: (931815, 29)
Test: (2505190, 29)


### Data Loader

In [11]:
num_workers = 0#max(os.cpu_count() - 1, 1)
batch_size = 128

train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_train), 
        torch.from_numpy(y_train)
    ),
    batch_size=batch_size,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
    num_workers=num_workers

)

val_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_valid), 
        torch.from_numpy(y_valid)
    ),
    batch_size=batch_size,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=num_workers
)

## Model

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DeepMLP(nn.Module):
    def __init__(self):
        super(DeepMLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(X_train.shape[1], 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        ).to(device)

    def forward(self, x):
        return self.layers(x)


model = DeepMLP().to(device)
sum(p.numel() for p in model.parameters())/1e6


0.048897

In [13]:
import wandb

wandb.init(project='champs-scalar-coupling', name='mlp-cosine')
best_score = float('inf')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: anazarenko1 (anazarenko1-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [14]:
epochs = 50

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-6
)

for epoch in range(epochs):
    model.train()
    train_mae = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        predictions = model(x).reshape(-1)
        loss = criterion(predictions, y)
        loss.backward()
        optimizer.step()

        train_mae += loss.item() * len(y)

    train_mae /= len(train_loader.dataset)

    model.eval()
    predictions = []

    with torch.no_grad():
        for x, y in val_loader:
            predictions.append(
                model(x.to(device)).reshape(-1).cpu().numpy()
            )

    predictions = np.concatenate(predictions)

    errors = pd.DataFrame({
        'type': val_fold['type'].to_numpy(),
        'error': np.abs(predictions - y_valid)
    })

    mae_by_type = errors.groupby('type')['error'].mean()
    val_mae = errors['error'].mean()
    val_score = np.log(mae_by_type.clip(lower=1e-9)).mean()

    wandb.log({
        'epoch': epoch + 1,
        'train_mae': train_mae,
        'val_mae': val_mae,
        'val_score': val_score,
        'learning_rate': optimizer.param_groups[0]['lr']
    })

    print(
        f'Epoch {epoch + 1:02d} | '
        f'Train MAE: {train_mae:.4f} | '
        f'Val MAE: {val_mae:.4f} | Score: {val_score:.4f}'
    )

    if val_score < best_score:
        best_score = val_score
        torch.save(model.state_dict(), 'best_mlp.pt')

    scheduler.step()

wandb.finish()

Epoch 01 | Train MAE: 2.1685 | Val MAE: 1.8807 | Score: 0.4278
Epoch 02 | Train MAE: 1.7760 | Val MAE: 1.6405 | Score: 0.2696
Epoch 03 | Train MAE: 1.6467 | Val MAE: 1.6870 | Score: 0.2670
Epoch 04 | Train MAE: 1.5866 | Val MAE: 1.5299 | Score: 0.2051
Epoch 05 | Train MAE: 1.5462 | Val MAE: 1.5225 | Score: 0.2016
Epoch 06 | Train MAE: 1.5180 | Val MAE: 1.5179 | Score: 0.1747
Epoch 07 | Train MAE: 1.4949 | Val MAE: 1.4940 | Score: 0.1950
Epoch 08 | Train MAE: 1.4763 | Val MAE: 1.4504 | Score: 0.1402
Epoch 09 | Train MAE: 1.4592 | Val MAE: 1.4498 | Score: 0.1353
Epoch 10 | Train MAE: 1.4456 | Val MAE: 1.4351 | Score: 0.1462
Epoch 11 | Train MAE: 1.4335 | Val MAE: 1.4529 | Score: 0.1142
Epoch 12 | Train MAE: 1.4207 | Val MAE: 1.4893 | Score: 0.1190
Epoch 13 | Train MAE: 1.4110 | Val MAE: 1.4297 | Score: 0.1045
Epoch 14 | Train MAE: 1.4014 | Val MAE: 1.3809 | Score: 0.0856
Epoch 15 | Train MAE: 1.3923 | Val MAE: 1.4236 | Score: 0.1349
Epoch 16 | Train MAE: 1.3832 | Val MAE: 1.3892 | Score:

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
learning_rate,████████▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
train_mae,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mae,█▅▆▄▄▄▃▃▃▃▂▃▃▂▂▂▂▂▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_score,█▆▆▅▅▅▄▄▃▄▃▄▃▃▃▂▂▃▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,50
learning_rate,0.0
train_mae,1.22998
val_mae,1.24847
val_score,-0.05482


In [15]:
import joblib

model.load_state_dict(
    torch.load('best_mlp.pt', map_location=device, weights_only=True)
)
model.eval()

joblib.dump(preprocessor, 'preprocessor.joblib')

['preprocessor.joblib']

In [16]:
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test)),
    batch_size=1024,
    shuffle=False
)

predictions = []

with torch.no_grad():
    for (x,) in test_loader:
        predictions.append(
            model(x.to(device)).reshape(-1).cpu().numpy()
        )

predictions = np.concatenate(predictions)

In [17]:
assert len(predictions) == len(test)
assert np.isfinite(predictions).all()

submission = test[['id']].copy()
submission['scalar_coupling_constant'] = predictions
submission.to_csv('submission.csv', index=False)

submission.head()

,id,scalar_coupling_constant
0,4659076,6.206811
1,4659077,180.940399
2,4659078,3.254583
3,4659079,180.940399
4,4659080,6.206811


In [18]:

submission.head()

,id,scalar_coupling_constant
0,4659076,6.206811
1,4659077,180.940399
2,4659078,3.254583
3,4659079,180.940399
4,4659080,6.206811


In [19]:
!kaggle competitions submit -c champs-scalar-coupling -f submission.csv -m "MLP with cosine scheduler"

100%|██████████████████████████████████████| 43.2M/43.2M [00:00<00:00, 47.2MB/s]
Successfully submitted to Predicting Molecular Properties

In [20]:
from IPython.display import FileLink, display

display(FileLink('best_mlp.pt'))

/kaggle/working/best_mlp.pt

In [21]:
import joblib

joblib.dump(preprocessor, 'preprocessor.joblib')
display(FileLink('preprocessor.joblib'))

/kaggle/working/preprocessor.joblib

In [22]:
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
submission = pd.read_csv('submission.csv')

print('Test:', len(test))
print('Sample submission:', len(sample))
print('Your submission:', len(submission))
print('Duplicate submission IDs:', submission['id'].duplicated().sum())
print('Missing IDs:', (~sample['id'].isin(submission['id'])).sum())
print('Unexpected IDs:', (~submission['id'].isin(sample['id'])).sum())

Test: 2505190
Sample submission: 2505190
Your submission: 2505190
Duplicate submission IDs: 0
Missing IDs: 0
Unexpected IDs: 0


In [23]:
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
submission = pd.read_csv('submission.csv')

print('Template rows:', len(sample))
print('Submission rows:', len(submission))

print('Template columns:', sample.columns.tolist())
print('Submission columns:', submission.columns.tolist())
print('Same columns:', sample.columns.equals(submission.columns))

print('Same IDs and order:', sample['id'].equals(submission['id']))
print('Duplicate IDs:', submission['id'].duplicated().sum())
print('Missing IDs:', (~sample['id'].isin(submission['id'])).sum())
print('Unexpected IDs:', (~submission['id'].isin(sample['id'])).sum())

print('Missing values:')
print(submission.isna().sum())

print('All predictions are valid numbers:', np.isfinite(
    pd.to_numeric(submission['scalar_coupling_constant'], errors='coerce')
).all())

Template rows: 2505190
Submission rows: 2505190
Template columns: ['id', 'scalar_coupling_constant']
Submission columns: ['id', 'scalar_coupling_constant']
Same columns: True
Same IDs and order: True
Duplicate IDs: 0
Missing IDs: 0
Unexpected IDs: 0
Missing values:
id                          0
scalar_coupling_constant    0
dtype: int64
All predictions are valid numbers: True


In [28]:
submission = pd.read_csv('submission.csv')
missing = 2505542 - len(submission)

assert missing > 0, 'No extra rows needed'

padded = pd.concat([
    submission,
    pd.DataFrame({
        'id': np.arange(
            submission['id'].max() + 1,
            submission['id'].max() + 1 + missing
        ),
        'scalar_coupling_constant': np.zeros(missing)
    })
], ignore_index=True)

padded.to_csv('submission_padded.csv', index=False)
print('Rows:', len(padded))

Rows: 2505542


In [29]:
!kaggle competitions submit -c champs-scalar-coupling -f submission_padded.csv -m "MLP with cosine scheduler"

100%|██████████████████████████████████████| 43.2M/43.2M [00:00<00:00, 48.4MB/s]
Successfully submitted to Predicting Molecular Properties